# Walkthrough: Adaptive Weak-Measurement Entanglement Estimation

This notebook demonstrates the simulation workflow, adaptive design behavior, and conformal calibration outputs.

In [ ]:
import numpy as np
from pathlib import Path

from src.data import StateParams, rho_ab
from src.design import estimate_state_with_policy
from src.metrics import concurrence, negativity
from src.weak_measurement import simulate_counts_for_state

## 1) Inspect the benchmark state family

In [ ]:
params = StateParams(p=0.8, theta=0.22)
rho = rho_ab(params)
print('trace=', np.trace(rho))
print('C, N =', concurrence(rho), negativity(rho))

## 2) Simulate one weak-measurement setting

In [ ]:
rng = np.random.default_rng(7)
counts, probs = simulate_counts_for_state(params, g=0.35, pointer_basis='X', shots=1000, rng=rng, noise='ideal')
print('Counts:', counts)
print('Prob sum:', sum(probs.values()))

## 3) Adaptive vs fixed estimation

In [ ]:
g_grid = [0.00, 0.10, 0.20, 0.35, 0.50, 0.70, 0.90]
out_fixed = estimate_state_with_policy(params, g_grid=g_grid, policy='fixed', shots=2000, seed=7, rounds=10, noise='ideal')
out_adapt = estimate_state_with_policy(params, g_grid=g_grid, policy='adaptive', shots=2000, seed=7, rounds=10, noise='ideal')
print('Fixed C_hat:', out_fixed['c_hat'])
print('Adaptive C_hat:', out_adapt['c_hat'])
print('Adaptive posterior entropy:', out_adapt['posterior_entropy'])

## 4) End-to-end artifact generation

Run from repository root:
`python -m src.main --mode sweep --backend sim --noise ideal --policy adaptive --g_grid 0.00,0.10,0.20,0.35,0.50,0.70,0.90 --shots 2000 --seed 7 --export_dir artifacts`

## 5) Optional hardware section

Set `QISKIT_IBM_TOKEN` and run `--mode hardware_run` to submit Runtime SamplerV2 jobs. Cached counts and job IDs are stored in `artifacts/ibm_cache/`.